In [ ]:
# =============================================================================
# USER-PROVIDED CALIBRATION PIPELINE
# =============================================================================
# This pipeline is for users who already have single-channel calibration files
# (not generated from manufacturer .cal/.xml files or the full pipeline).
#
# Workflow:
# 1. User places their single-channel .yml calibration files in a folder
# 2. User provides the path to their raw files (or pre-extracted raw configs)
# 3. Pipeline loads both and runs the mapping algorithm
# 4. Outputs: mapping dictionary + calibration configurations
#
# The single-channel files must each be a flat YAML dictionary with the same
# fields used in the full pipeline's single-channel output format.
# File names can be anything as long as they are unique and end with .yml.
#
# Key matching fields (must be present and correct for mapping to work):
#   - transceiver_id, transducer_model, transducer_serial_number
#   - pulse_form, frequency_start, frequency_end
#   - transmit_power, transmit_duration_nominal
# =============================================================================

from pathlib import Path

# Raw file reading
from aa_si_calibration.raw_reader_api import process_raw_folder, save_yaml

# Mapping algorithm and pipeline functions
from aa_si_calibration.mapping_algorithm import (
    load_raw_configs,
    load_calibration_data_from_single_files,
    build_mapping,
    get_calibration,
    get_calibration_from_file,
    save_mapping_files,
    print_mapping_preview,
    handle_unused_calibration_files,
    check_for_conflicts,
    set_record_author,
)

print("All imports successful!")

All imports successful!


In [ ]:
# =============================================================================
# CONFIGURATION - Define Input and Output Paths
# =============================================================================

# If True, unused calibration files (those that don't match any raw channel)
# are deleted from the calibration folder during the mapping step.
DELETE_UNUSED_STANDARDIZED_CALIBRATION_FILES = True

# Record author - the individual running this pipeline and generating the records
RECORD_AUTHOR = "Placeholder Author"

# --- INPUT: Single-channel calibration files ---
# Point this to a folder containing your single-channel .yml calibration files.
# Each file should be a flat YAML dict for one channel.
CAL_FILES_FOLDER = Path("./example_data/ek60_single_channel_yml_cal_files_input")

# --- INPUT: Raw files ---
# Option A: Provide a folder of .raw files to extract configs from
RAW_INPUT_FOLDER = Path("./example_data/ek60_raw_file_input_folder")
# RAW_INPUT_FOLDER = Path("./example_data/ek80_CW_raw_file_input_folder")

# Option B: Provide a pre-extracted raw_file_configs.yaml
# RAW_CONFIGS_PATH = Path("./Full_Pipeline_Outputs/raw_file_configs/raw_file_configs.yaml")

# --- OUTPUT ---
OUTPUT_BASE = Path("./User_Provided_Cal_Pipeline_Outputs")
RAW_CONFIGS_OUTPUT = OUTPUT_BASE / "raw_file_configs"
MAPPING_OUTPUT = OUTPUT_BASE / "mapping_files"

for folder in [RAW_CONFIGS_OUTPUT, MAPPING_OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

RAW_CONFIGS_PATH = RAW_CONFIGS_OUTPUT / "raw_file_configs.yaml"

print(f"Record author: {RECORD_AUTHOR}")
print(f"\nInput folders:")
print(f"  Calibration files: {CAL_FILES_FOLDER}")
print(f"  Raw files:         {RAW_INPUT_FOLDER}")
print(f"\nOutput folder:       {OUTPUT_BASE}")

Record author: Placeholder Author

Input folders:
  Calibration files: example_data\ek60_single_channel_yml_cal_files_input
  Raw files:         example_data\ek60_raw_file_input_folder

Output folder:       User_Provided_Cal_Pipeline_Outputs


In [ ]:
# =============================================================================
# STEP 1: READ RAW FILE CONFIGURATIONS
# =============================================================================
# Process all .raw files in the input folder and extract channel configurations.
# If you already have a raw_file_configs.yaml, you can skip this cell and
# uncomment the RAW_CONFIGS_PATH in the configuration cell above.

file_configs, frequencies_set = process_raw_folder(RAW_INPUT_FOLDER, verbose=True)

# Save raw file configurations to YAML
save_yaml(file_configs, RAW_CONFIGS_PATH)
print(f"\nSaved raw file configurations to: {RAW_CONFIGS_PATH}")

Found 3 raw files in example_data\ek60_raw_file_input_folder
  - D20160725-T205832.raw
  - D20160725-T212129.raw
  - D20160725-T214425.raw
File: D20160725-T205832.raw
Instrument (detected): EK60
File format (from reader): EK60

--- GPS Summary ---
  NMEA datagrams found: 3896
  Valid GPS fixes: 2064
  First GPS: 41.745935, -65.507845
File: D20160725-T212129.raw
Instrument (detected): EK60
File format (from reader): EK60

--- GPS Summary ---
  NMEA datagrams found: 3921
  Valid GPS fixes: 2064
  First GPS: 41.813062, -65.503617
File: D20160725-T214425.raw
Instrument (detected): EK60
File format (from reader): EK60

--- GPS Summary ---
  NMEA datagrams found: 3808
  Valid GPS fixes: 2066
  First GPS: 41.881765, -65.499538

SUMMARY: Processed 3 files (sorted by metadata_start_time)
Unique frequencies found: [18000.0, 38000.0, 70000.0, 120000.0, 200000.0] Hz

 Saved raw file configurations to: User_Provided_Cal_Pipeline_Outputs\raw_file_configs\raw_file_configs.yaml


In [4]:
# =============================================================================
# STEP 2: LOAD RAW FILE CONFIGURATIONS
# =============================================================================
# Load the raw file configurations we saved in Step 1.
# Calibration data is loaded in the next cell (together with the mapping step)
# so that deleting calibration files and re-running that cell is sufficient.

raw_file_configs = load_raw_configs(RAW_CONFIGS_PATH)

print(f"Loaded {len(raw_file_configs)} raw file configurations")
print(f"\nRaw files: {[f['filename'] for f in raw_file_configs]}")

Loaded 3 raw file configurations

Raw files: ['D20160725-T205832.raw', 'D20160725-T212129.raw', 'D20160725-T214425.raw']


In [5]:
# =============================================================================
# STEP 3: LOAD CALIBRATION DATA AND BUILD MAPPING
# =============================================================================
# Loads the single-channel calibration files, then matches raw channels to
# calibration data.  If any raw channel matches MULTIPLE calibration files,
# this cell prints the conflicting files and raises an error.
#
# To resolve: delete the unwanted calibration file(s) from the
# calibration files folder and RE-RUN THIS CELL.

# Load calibration data from the single-channel files directory
calibration_data = load_calibration_data_from_single_files(CAL_FILES_FOLDER)

print(f"Loaded {len(calibration_data['channels'])} calibration channel(s) "
      f"from {CAL_FILES_FOLDER}")

# Set record_author on calibration channels if not already present
set_record_author(calibration_data, RECORD_AUTHOR)

# Build the mapping
result = build_mapping(raw_file_configs, calibration_data, verbose=False)
result.print_summary()

# Delete unused calibration files (before conflict resolution)
if DELETE_UNUSED_STANDARDIZED_CALIBRATION_FILES:
    handle_unused_calibration_files(
        result, calibration_data, CAL_FILES_FOLDER,
        keep_unused=False,
    )

# Check for multiple matches (raises ValueError if conflicts exist)
check_for_conflicts(result, cal_files_dir=CAL_FILES_FOLDER)

# Access the dictionaries from the result
mapping_dict = result.mapping_dict
calibration_dict = result.calibration_dict

Loaded 5 calibration channel(s) from example_data\ek60_single_channel_yml_cal_files_input

MATCHING SUMMARY

Raw file channels:
  Total file channels processed: 15
  Total unique channels: 5
  Matched file channels: 15
  Matched unique channels: 5
  Unmatched file channels: 0
  Multiplexing warnings: 0

Calibration files:
  Total calibrations loaded: 5
  Unique calibrations used: 5
  Conflicting calibration sets: 0


In [ ]:
# =============================================================================
# STEP 4: SAVE MAPPING FILES AND PREVIEW
# =============================================================================

# Preview the mapping results
print_mapping_preview(result)

# Save the mapping and calibration dictionaries
mapping_path, calibration_path = save_mapping_files(result, MAPPING_OUTPUT)

print(f"\nSaved mapping dictionary to: {mapping_path}")
print(f"Saved calibration dictionary to: {calibration_path}")

MAPPING DICTIONARY PREVIEW

D20160725-T205832.raw:
  GPT  38 kHz 0090720346bc 1-1 ES38B
    -> 2016-07-18_GPT  38 kHz
  GPT  18 kHz 009072056b0e 2-1 ES18-11
    -> 2016-07-18_GPT  18 kHz
  GPT 120 kHz 0090720580f1 3-1 ES120-7
    -> 2016-07-18_GPT 120 kHz
  GPT 200 kHz 009072034261 4-1 ES200-7
    -> 2016-07-18_GPT 200 kHz
  GPT  70 kHz 00907206042f 5-1 ES70-7C
    -> 2016-07-18_GPT  70 kHz

D20160725-T212129.raw:
  GPT  38 kHz 0090720346bc 1-1 ES38B
    -> 2016-07-18_GPT  38 kHz
  GPT  18 kHz 009072056b0e 2-1 ES18-11
    -> 2016-07-18_GPT  18 kHz
  GPT 120 kHz 0090720580f1 3-1 ES120-7
    -> 2016-07-18_GPT 120 kHz
  GPT 200 kHz 009072034261 4-1 ES200-7
    -> 2016-07-18_GPT 200 kHz
  GPT  70 kHz 00907206042f 5-1 ES70-7C
    -> 2016-07-18_GPT  70 kHz

D20160725-T214425.raw:
  GPT  38 kHz 0090720346bc 1-1 ES38B
    -> 2016-07-18_GPT  38 kHz
  GPT  18 kHz 009072056b0e 2-1 ES18-11
    -> 2016-07-18_GPT  18 kHz
  GPT 120 kHz 0090720580f1 3-1 ES120-7
    -> 2016-07-18_GPT 120 kHz
  GPT 200 

In [7]:
# =============================================================================
# EXAMPLE: Retrieve calibration for a specific raw file + channel
# =============================================================================

print("Available raw files and channels:")
for filename, channels in mapping_dict.items():
    print(f"\n  {filename}:")
    for channel_id, cal_key in channels.items():
        print(f"    - {channel_id}")
        print(f"      -> maps to: {cal_key}")

# Retrieve calibration for the first file/channel as an example
if mapping_dict:
    example_filename = list(mapping_dict.keys())[0]
    if mapping_dict[example_filename]:
        example_channel = list(mapping_dict[example_filename].keys())[0]
        
        cal_data = get_calibration(example_filename, example_channel, mapping_dict, calibration_dict)
        
        if cal_data:
            print(f"\nCalibration for '{example_filename}' -> '{example_channel}':")
            print(f"  Calibration date: {cal_data.get('calibration_date')}")
            print(f"  Frequency: {cal_data.get('frequency')}")
            print(f"  Gain correction: {cal_data.get('gain_correction')}")
            print(f"  Sa correction: {cal_data.get('sa_correction')}")
            print(f"  Equivalent beam angle: {cal_data.get('equivalent_beam_angle')}")
            print(f"  Sound speed: {cal_data.get('sound_speed_indicative')}")
            print(f"  Absorption: {cal_data.get('absorption_indicative')}")
        else:
            print(f"No calibration found for {example_filename} -> {example_channel}")

Available raw files and channels:

  D20160725-T205832.raw:
    - GPT  38 kHz 0090720346bc 1-1 ES38B
      -> maps to: 2016-07-18_GPT  38 kHz
    - GPT  18 kHz 009072056b0e 2-1 ES18-11
      -> maps to: 2016-07-18_GPT  18 kHz
    - GPT 120 kHz 0090720580f1 3-1 ES120-7
      -> maps to: 2016-07-18_GPT 120 kHz
    - GPT 200 kHz 009072034261 4-1 ES200-7
      -> maps to: 2016-07-18_GPT 200 kHz
    - GPT  70 kHz 00907206042f 5-1 ES70-7C
      -> maps to: 2016-07-18_GPT  70 kHz

  D20160725-T212129.raw:
    - GPT  38 kHz 0090720346bc 1-1 ES38B
      -> maps to: 2016-07-18_GPT  38 kHz
    - GPT  18 kHz 009072056b0e 2-1 ES18-11
      -> maps to: 2016-07-18_GPT  18 kHz
    - GPT 120 kHz 0090720580f1 3-1 ES120-7
      -> maps to: 2016-07-18_GPT 120 kHz
    - GPT 200 kHz 009072034261 4-1 ES200-7
      -> maps to: 2016-07-18_GPT 200 kHz
    - GPT  70 kHz 00907206042f 5-1 ES70-7C
      -> maps to: 2016-07-18_GPT  70 kHz

  D20160725-T214425.raw:
    - GPT  38 kHz 0090720346bc 1-1 ES38B
      -> ma

In [ ]:
# =============================================================================
# PIPELINE SUMMARY
# =============================================================================

print("=" * 80)
print("PIPELINE COMPLETE")
print("=" * 80)


def list_files_recursive(folder, indent=0):
    """List all files in a folder recursively."""
    if not folder.exists():
        return
    for item in sorted(folder.iterdir()):
        if item.is_dir():
            print("  " * indent + f" {item.name}/")
            list_files_recursive(item, indent + 1)
        else:
            size_kb = item.stat().st_size / 1024
            print("  " * indent + f" {item.name} ({size_kb:.1f} KB)")


print(f"\nOutput directory: {OUTPUT_BASE}")
print("-" * 40)
list_files_recursive(OUTPUT_BASE)

print("\n" + "=" * 80)
print("OUTPUT FILES")
print("=" * 80)
print("""
1. raw_file_configs.yaml - Channel configurations extracted from raw files
2. channel_to_calibration_mapping.yaml - Maps each raw file + channel to a cal key
3. calibration_configurations.yaml - Calibration parameters indexed by key

Use the mapping files with get_calibration() or get_calibration_from_file()
to retrieve calibration data for any raw file + channel combination.
""")


Output directory: User_Provided_Cal_Pipeline_Outputs
----------------------------------------
📁 mapping_files/
  📄 calibration_configurations.yaml (8.7 KB)
  📄 channel_to_calibration_mapping.yaml (1.0 KB)
📁 raw_file_configs/
  📄 raw_file_configs.yaml (11.2 KB)

PIPELINE COMPLETE

Output files produced:

1. raw_file_configs.yaml - Channel configurations extracted from raw files
2. channel_to_calibration_mapping.yaml - Maps each raw file + channel to a cal key
3. calibration_configurations.yaml - Calibration parameters indexed by key

Use the mapping files with get_calibration() or get_calibration_from_file()
to retrieve calibration data for any raw file + channel combination.

